# Adaptive RAG Routing System
_Purpose: Introduce the system and list available routes._
This notebook implements an adaptive Retrieval-Augmented Generation (RAG) system with intelligent query routing that directs different types of queries to different retrieval and processing paths.

## System Overview
- **Route 0**: Irrelevant queries - Use template-based responses
- **Route 1**: Generic company-related queries - Retrieve from company metadata
- **Route 2**: Product queries - Use semantic search with metadata filters

#### Imports

In [106]:
import json
import re
from typing import Dict, List, Tuple, Optional, Any
from enum import Enum
from dataclasses import dataclass
import pandas as pd
from pathlib import Path

#### Helpers

In [107]:
def banner(title, pad='----', width=21):
    """Print a formatted banner."""
    print(pad * width)
    print(title)
    print(pad * width)

In [108]:
# Enum for query route types
class QueryRoute(Enum):
    IRRELEVANT = 0
    COMPANY_RELATED = 1
    PRODUCT_QUERY = 2

# Flag indicating if a route applies to the query
@dataclass # Generates special methods, making it easier to create simple data-holding classes
class RouteFlag:
    route: QueryRoute
    applies: bool
    confidence: float
    reasoning: str

# Multi-route flagging result - indicates which routes apply
@dataclass
class RoutingFlags:
    query: str
    flags: Dict[QueryRoute, RouteFlag]
    primary_route: QueryRoute
    secondary_routes: List[QueryRoute]
    total_applicable: int

#### Initial data and configuration loading step

In [109]:
# Load Route 0: Irrelevant Query Template
with open("company_irrelevant_template_reply.json") as f:
    # Used later when primary_route == QueryRoute.IRRELEVANT to generate replies.
    irrelevant_config = json.load(f) # Loads it as a Python dictionary

# Load Route 1: Company Information
with open("company_info.json") as f:
    company_info = json.load(f) # Loads it as a Python dictionary

# Load Route 2: Fashion/Product Data
try:
    fashion_df = pd.read_csv("fashion_with_brands.csv")
    print(f"◉ Loaded {len(fashion_df)} products from fashion_with_brand.csv")
except FileNotFoundError:
    print("⚠ fashion_with_brand.csv not found - product queries will have limited data")
    fashion_df = None

print("◉ Configuration loaded successfully!")

# Load domain vocabulary for product keywords
with open("domain_vocabulary.json") as f:
    domain_vocab = json.load(f)

# Extract all product-related keywords from domain vocabulary
def extract_product_keywords_from_vocab(domain_vocab: dict) -> list:
    """Dynamically extract product-related keywords from domain_vocabulary.json"""
    keywords = set()
    
    # Extract from all categories in domain vocabulary
    for category, terms in domain_vocab.items():
        if isinstance(terms, list):
            keywords.update(term.lower() for term in terms)
    
    # Adds a small list of verbs that are strong indicators of shopping intent 
    keywords.update(["product", "buy", "sell", "price", "purchase", "shop", "shopping"])
    
    return sorted(list(keywords))

PRODUCT_KEYWORDS = extract_product_keywords_from_vocab(domain_vocab)
print(f"◉ Extracted {len(PRODUCT_KEYWORDS)} product keywords from domain vocabulary")

# Extract company-related keywords from company_info.json
def extract_company_keywords_from_config(company_config: dict) -> list:
    """Dynamically extract company-related keywords from company_info.json"""
    keywords = set()
    
    # Extract keywords from the structure keys and descriptions
    company_data = company_config.get("company_data", {})
    for key, metadata in company_data.items():
        # Add the key name (split by underscore)
        keywords.update(key.split('_'))
        
        # Add words from the key field
        if "key" in metadata:
            keywords.add(metadata["key"].lower())
        
        # Extract relevant words from description
        if "description" in metadata:
            desc_words = metadata["description"].lower().split()
            # Filter for relevant nouns/keywords (simple heuristic)
            relevant_words = [w.strip('.,') for w in desc_words if len(w) > 3]
            keywords.update(relevant_words[:5])  # Take first few words from each description
    
    # Add common company-related terms
    keywords.update([
        "company", "contact", "location", "founder", "about", 
        "phone", "email", "store", "address", "hours", 
        "opening", "closing", "customer", "care", "support",
        "office", "branch", "headquarter", "team"
    ])
    
    return sorted(list(keywords))

COMPANY_KEYWORDS = extract_company_keywords_from_config(company_info)
print(f"◉ Extracted {len(COMPANY_KEYWORDS)} company keywords from configuration")

◉ Loaded 2906 products from fashion_with_brand.csv
◉ Configuration loaded successfully!
◉ Extracted 278 product keywords from domain vocabulary
◉ Extracted 51 company keywords from configuration


## Query Processing Pipeline
_Purpose: Process queries through complexity detection and routing._

### Step 1: Complexity Detection & Query Splitting
Query complexity indicators for detecting multi-part queries

In [110]:
# Query Processing Pipeline: Split complex queries and determine routes for each

# Step 1: Complexity Detection & Query Splitting

# Query complexity indicators for detecting multi-part queries
COMPLEXITY_INDICATORS = [
    r'and|or|also|as well as|additionally|plus',
    r'\?.*\?',  # Multiple question marks
]

# Detect if query contains multiple parts/questions
def detect_query_complexity(user_query: str) -> Tuple[bool, int]:
    query_lower = user_query.lower()
    complexity_score = sum(1 for pattern in COMPLEXITY_INDICATORS if re.search(pattern, query_lower))
    return complexity_score > 0, complexity_score

# Split complex queries into simpler subqueries
def split_query(user_query: str) -> List[str]:
    is_complex, _ = detect_query_complexity(user_query)
    if not is_complex:
        return [user_query.strip()]

    split_pattern = r"\b(?:and|also|plus|as well as|additionally)\b|\?+"
    parts = [part.strip(" ?") for part in re.split(split_pattern, user_query) if part.strip(" ?")]
    return parts if parts else [user_query.strip()]

# Step 2: Route Determination for Individual Queries

# Determine the primary route for a single query
def determine_route(query: str) -> QueryRoute:
    query_lower = query.lower()
    company_score = sum(1 for kw in COMPANY_KEYWORDS if kw in query_lower)
    product_score = sum(1 for kw in PRODUCT_KEYWORDS if kw in query_lower)
    
    # Route determination priority
    if company_score > 0 and product_score == 0:
        return QueryRoute.COMPANY_RELATED
    elif product_score > 0 and company_score == 0:
        return QueryRoute.PRODUCT_QUERY
    elif company_score > 0 and product_score > 0:
        # Both match, prefer product for specificity
        return QueryRoute.PRODUCT_QUERY
    else:
        # No keywords match
        return QueryRoute.IRRELEVANT

# Helper functions for company metadata handling

def extract_company_metadata_keys() -> Dict[str, Dict[str, Any]]:
    company_data = company_info["company_data"]
    schema = {}
    for key, metadata in company_data.items():
        schema[key] = {
            "key": metadata.get("key", key),
            "description": metadata.get("description", ""),
            "value": metadata.get("value", "")
        }
    return schema


In [111]:
# Route handlers - simplified for sequential per-query routing

## Route 0: Irrelevant Queries
def route_0_irrelevant_query(query: str, username: str = "Customer") -> Dict[str, Any]:
    """
    Route 0: Irrelevant Query.
    Returns query + template JSON answer.
    
    Returns:
        Dict with query and template response
    """
    route_data = irrelevant_config["company_data"]
    # Get template
    template = route_data["fallback_response"]["value"]
    # Format template with username
    response_text = template.replace("{{username}}", username)
    
    return {
        "query": query,
        "route": QueryRoute.IRRELEVANT.name,
        "response": response_text
    }

## Route 1: Company Queries
def route_1_company_query(query: str) -> Dict[str, Any]:
    """
    Route 1: Company Query.
    Matches keywords/metadata from query to JSON file, returns matched data with values.
    
    Returns:
        Dict with query and matched company metadata (keys and values)
    """
    company_data = company_info["company_data"]
    query_lower = query.lower()
    matched_metadata = {}
    
    # Match keywords to company metadata keys
    for key, metadata in company_data.items():
        key_text = metadata.get("key", "").lower()
        description = metadata.get("description", "").lower()
        # Check if query words match this metadata key
        if any(word in query_lower for word in key.split('_')) or \
           any(word in query_lower for word in key_text.split()) or \
           any(word in query_lower for word in description.split()[:5]):
            matched_metadata[key] = metadata["value"]
    
    # If no specific matches, return all company data as available
    if not matched_metadata:
        matched_metadata = {k: v["value"] for k, v in company_data.items()}
    
    return {
        "query": query,
        "route": QueryRoute.COMPANY_RELATED.name,
        "matched_metadata": matched_metadata
    }

## Route 2: Product Queries
def route_2_product_query(query: str) -> Dict[str, Any]:
    """
    Route 2: Product Query.
    Matches keywords/metadata to predefined lists, returns query + applicable filters.
    """
    filters = _extract_domain_filters(query)
    
    return {
        "query": query,
        "route": QueryRoute.PRODUCT_QUERY.name,
        "metadata_filters": filters
    }


In [112]:
## Product Query Helper Functions

# Split complex product queries into parts using simple conjunction rules
def _split_product_queries(user_query: str) -> List[str]:
    split_pattern = r"\b(?:and|also|plus|as well as|additionally)\b|\?+"
    parts = [part.strip(" ?") for part in re.split(split_pattern, user_query) if part.strip(" ?")]
    return parts if parts else [user_query.strip()]

# Extract matching terms from domain vocabulary for lightweight filtering
def _extract_domain_filters(user_query: str) -> Dict[str, List[str]]:
    query_lower = user_query.lower()
    filters = {}
    for key, terms in domain_vocab.items():
        if isinstance(terms, list):
            matches = [term for term in terms if term.lower() in query_lower]
            if matches:
                filters[key] = sorted(set(matches))
    return filters

In [113]:
# Purpose: Orchestrate the full routing pipeline following the exact 4-step sequence.

def adaptive_rag_router(user_query: str, username: str = "Customer") -> Dict[str, Any]:
    """
    Main Adaptive RAG Routing orchestrator with sequential per-query processing.
    
    Required Pipeline Sequence:
    1. Query is entered
    2. If it's complex, it's broken down into simpler queries
    3. For each one of the simple queries, determine the route
    4. Call route-specific handler and assemble results
        
    Returns:
        Complete response with subquery results
    """
    
    # Step 1: Query entered (already done by function parameter)
    
    # Step 2: Check complexity and split if needed
    subqueries = split_query(user_query)
    
    # Step 3: For each subquery, determine route
    subquery_results = []
    for subquery in subqueries:
        route = determine_route(subquery)
        
        # Step 4: Call route-specific handler
        if route == QueryRoute.IRRELEVANT:
            response = route_0_irrelevant_query(subquery, username)
        elif route == QueryRoute.COMPANY_RELATED:
            response = route_1_company_query(subquery)
        elif route == QueryRoute.PRODUCT_QUERY:
            response = route_2_product_query(subquery)
        else:
            # Fallback (shouldn't happen)
            response = route_0_irrelevant_query(subquery, username)
        
        subquery_results.append(response)
    
    # Assemble final response
    return {
        "original_query": user_query,
        "subqueries": subqueries,
        "results": subquery_results,
        "pipeline_status": "success"
    }

print("◉ Adaptive RAG Router initialized\n")

◉ Adaptive RAG Router initialized



In [114]:
# Print simplified subqueries when the original query is complex, then route normally
def inspect_complexity_and_route(user_query: str, username: str = "Customer") -> Dict[str, Any]:
    subqueries = split_query(user_query)
    
    if len(subqueries) > 1:
        banner("Complex Query Detected")
        print(f"Original query: {user_query}")
        print(f"Simplified into {len(subqueries)} queries:")
        for idx, subquery in enumerate(subqueries, 1):
            print(f"  {idx}. {subquery}")
        print("")
    
    return adaptive_rag_router(user_query, username)

## End-to-End Testing
_Purpose: Introduce the test suite._
Comprehensive demonstration of the adaptive RAG routing pipeline with different query types.

In [115]:
def print_adaptive_routing_result(result: Dict[str, Any], test_num: int):
    """Pretty print adaptive routing results with per-subquery routing"""
    banner(f"TEST {test_num}: {result['original_query']}", width=60)
    
    if len(result['subqueries']) > 1:
        print(f"\n📝 SUBQUERY BREAKDOWN:")
        for idx, sq in enumerate(result['subqueries'], 1):
            print(f"   {idx}. {sq}")
    
    print(f"\n📊 ROUTING RESULTS:")
    for idx, res in enumerate(result['results'], 1):
        print(f"\n   Query {idx}: {res['query']}")
        print(f"   Route: {res['route']}")
        
        if res['route'] == 'IRRELEVANT':
            print(f"   Response: {res['response']}")
        elif res['route'] == 'COMPANY_RELATED':
            print(f"   Matched Metadata:")
            for key, value in res['matched_metadata'].items():
                print(f"      - {key}: {value}")
        elif res['route'] == 'PRODUCT_QUERY':
            print(f"   Metadata Filters: {res['metadata_filters']}")
    
    print(f"\n✅ Pipeline Status: {result['pipeline_status']}\n")

# Test Suite
banner("🚀 ADAPTIVE RAG ROUTING SYSTEM - SEQUENTIAL PER-QUERY PIPELINE")

# Test 1: Irrelevant Query
result_1 = inspect_complexity_and_route("What's the weather like today?", "Sarah")
print_adaptive_routing_result(result_1, 1)

# Test 2: Company-Related Query
result_2 = inspect_complexity_and_route("Can you tell me where your stores are located?")
print_adaptive_routing_result(result_2, 2)

# Test 3: Simple Product Query
result_3 = inspect_complexity_and_route("I'm looking for white tops for girls in large size")
print_adaptive_routing_result(result_3, 3)

# Test 4: Complex Multi-Query (Product + Company)
result_4 = inspect_complexity_and_route("Show me blue tops for girls and what's your contact info?")
print_adaptive_routing_result(result_4, 4)

# Test 5: Company Info Query
result_5 = inspect_complexity_and_route("What's your contact information and store locations?")
print_adaptive_routing_result(result_5, 5)

# Test 6: Ambiguous Query
result_6 = inspect_complexity_and_route("I need something")
print_adaptive_routing_result(result_6, 6)

banner("✅ All tests completed successfully!")


------------------------------------------------------------------------------------
🚀 ADAPTIVE RAG ROUTING SYSTEM - SEQUENTIAL PER-QUERY PIPELINE
------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
TEST 1: What's the weather like today?
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

📊 ROUTING RESULTS:

   Query 1: What's the weather like today?
   Route: IRRELEVANT
   Response: Sorry Sarah, I cannot help you with your request. At Fashion Hub, we specialize in high-quality fashion products... we will connect you to a shop cus

## LangGraph Implementation
_Purpose: Graph-based orchestration of the routing pipeline._

This section implements a LangGraph state machine with:
- **Decision Node**: Determines which route to apply for each query
- **Route 0 Node**: Handles irrelevant queries
- **Route 1 Node**: Processes company-related queries  
- **Route 2 Node**: Executes product queries

In [116]:
# LangGraph imports
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

# Define the state schema for the graph
class RouterState(TypedDict):
    """State schema for the routing graph."""
    query: str
    route: Optional[QueryRoute]
    response: Optional[Dict[str, Any]]
    username: Optional[str]

In [117]:
# Node 1: Decision Node - determines which route to use
def decision_node(state: RouterState) -> RouterState:
    """Determine which route applies to the query."""
    query = state["query"]
    route = determine_route(query)
    state["route"] = route
    print(f"🔀 Decision Node: Query routed to {route.name} (Route {route.value})")
    return state

# Node 2: Route 0 Node - handles irrelevant queries
def route_0_node(state: RouterState) -> RouterState:
    """Process irrelevant queries with template responses."""
    query = state["query"]
    username = state.get("username", "User")
    response = route_0_irrelevant_query(query, username)
    state["response"] = response
    print(f"🚫 Route 0 Node: Generated template response")
    return state

# Node 3: Route 1 Node - handles company-related queries
def route_1_node(state: RouterState) -> RouterState:
    """Process company-related queries."""
    query = state["query"]
    response = route_1_company_query(query)
    state["response"] = response
    print(f"🏢 Route 1 Node: Retrieved company metadata")
    return state

# Node 4: Route 2 Node - handles product queries
def route_2_node(state: RouterState) -> RouterState:
    """Process product queries with metadata filters."""
    query = state["query"]
    response = route_2_product_query(query)
    state["response"] = response
    print(f"🛍️ Route 2 Node: Extracted product filters")
    return state

In [118]:
# Build the routing graph
def create_routing_graph() -> StateGraph:
    """Create and compile the LangGraph routing state machine."""
    
    # Initialize the graph
    workflow = StateGraph(RouterState)
    
    # Add all nodes
    workflow.add_node("decision", decision_node)
    workflow.add_node("route_0", route_0_node)
    workflow.add_node("route_1", route_1_node)
    workflow.add_node("route_2", route_2_node)
    
    # Set entry point
    workflow.set_entry_point("decision")
    
    # Define conditional routing function
    def route_decision(state: RouterState) -> Literal["route_0", "route_1", "route_2"]:
        """Route to appropriate node based on decision."""
        route = state["route"]
        if route == QueryRoute.IRRELEVANT:
            return "route_0"
        elif route == QueryRoute.COMPANY_RELATED:
            return "route_1"
        elif route == QueryRoute.PRODUCT_QUERY:
            return "route_2"
        else:
            raise ValueError(f"Unknown route: {route}")
    
    # Add conditional edges from decision node to route nodes
    workflow.add_conditional_edges(
        "decision",
        route_decision,
        {
            "route_0": "route_0",
            "route_1": "route_1",
            "route_2": "route_2"
        }
    )
    
    # Add edges from route nodes to END
    workflow.add_edge("route_0", END)
    workflow.add_edge("route_1", END)
    workflow.add_edge("route_2", END)
    
    # Compile the graph
    graph = workflow.compile()
    print("✅ Routing graph compiled successfully!")
    return graph

# Create the graph
routing_graph = create_routing_graph()

✅ Routing graph compiled successfully!


In [119]:
# Test the LangGraph routing
banner("LangGraph Test Execution", pad='====')

# Test with a product query
test_query = "Show me red dresses under $50"
print(f"\n📝 Input Query: '{test_query}'\n")

# Execute the graph
initial_state = {
    "query": test_query,
    "route": None,
    "response": None,
    "username": "Alice"
}

# Invoke the graph
final_state = routing_graph.invoke(initial_state)

# Display results
print("\n" + "=" * 80)
print("📊 FINAL STATE")
print("=" * 80)
print(f"Query: {final_state['query']}")
print(f"Route: {final_state['route'].name} (Route {final_state['route'].value})")
print(f"\nResponse:")
print(json.dumps(final_state['response'], indent=2))
print("=" * 80)

LangGraph Test Execution

📝 Input Query: 'Show me red dresses under $50'

🔀 Decision Node: Query routed to PRODUCT_QUERY (Route 2)
🛍️ Route 2 Node: Extracted product filters

📊 FINAL STATE
Query: Show me red dresses under $50
Route: PRODUCT_QUERY (Route 2)

Response:
{
  "query": "Show me red dresses under $50",
  "route": "PRODUCT_QUERY",
  "metadata_filters": {
    "colors": [
      "red"
    ],
    "products": [
      "dresses"
    ]
  }
}


In [120]:
# Test all routes with the LangGraph
banner("Testing All Routes with LangGraph", pad='====')

test_queries = [
    ("What's the weather today?", "Bob"),           # Route 0
    ("What is your company's mission?", "Carol"),   # Route 1  
    ("Find blue shirts for men", "Dave")            # Route 2
]

for idx, (query, username) in enumerate(test_queries, 1):
    print(f"\n{'─' * 80}")
    print(f"Test {idx}: '{query}'")
    print('─' * 80)
    
    state = routing_graph.invoke({
        "query": query,
        "route": None,
        "response": None,
        "username": username
    })
    
    print(f"\n✅ Route: {state['route'].name} (Route {state['route'].value})")
    print(f"📄 Response: {state['response']}")
    print()

banner("✅ All LangGraph tests completed!", pad='====')

Testing All Routes with LangGraph

────────────────────────────────────────────────────────────────────────────────
Test 1: 'What's the weather today?'
────────────────────────────────────────────────────────────────────────────────
🔀 Decision Node: Query routed to IRRELEVANT (Route 0)
🚫 Route 0 Node: Generated template response

✅ Route: IRRELEVANT (Route 0)
📄 Response: {'query': "What's the weather today?", 'route': 'IRRELEVANT', 'response': 'Sorry Bob, I cannot help you with your request. At Fashion Hub, we specialize in high-quality fashion products... we will connect you to a shop customer service representative.'}


────────────────────────────────────────────────────────────────────────────────
Test 2: 'What is your company's mission?'
────────────────────────────────────────────────────────────────────────────────
🔀 Decision Node: Query routed to COMPANY_RELATED (Route 1)
🏢 Route 1 Node: Retrieved company metadata

✅ Route: COMPANY_RELATED (Route 1)
📄 Response: {'query': "What 